# LeetCode 3956 — Maximum Sum of M Non-Overlapping Subarrays I bằng Python

- Link: https://leetcode.com/problems/maximum-sum-of-m-non-overlapping-subarrays-i/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP) phân đoạn, tổng tiền tố, cửa sổ trượt với hàng đợi đơn điệu (monotonic deque)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumSum(self, nums: List[int], m: int, l: int, r: int) -> int:
```

> Bài này là phiên bản **I** (`n ≤ 1000`). Phiên bản **II** (`n ≤ 10^5`, `|nums[i]| ≤ 10^5`) là một bài riêng với notebook riêng (LeetCode 3957) và cần kỹ thuật khác (Aliens trick).

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums` và ba số `m`, `l`, `r`. Chọn **ít nhất một và nhiều nhất `m`** mảng con **không giao nhau**, mỗi mảng con có độ dài trong `[l, r]`, sao cho **tổng các tổng** lớn nhất. Đây là bài DP phân đoạn kinh điển: "chia mảng thành các đoạn" kèm các ràng buộc về độ dài và số đoạn.

Với `n ≤ 1000` và `m ≤ n`, một bảng DP `n × m` (`10^6` ô) là vừa. Điều cần nghĩ kỹ là cách tính mỗi ô: nếu mỗi ô duyệt mọi độ dài đoạn cuối `[l, r]` thì tổng `O(n · m · r)` quá chậm; ta dùng **tổng tiền tố** để biến "tổng đoạn" thành hiệu hai số, rồi dùng **hàng đợi đơn điệu** lấy giá trị lớn nhất trong một **cửa sổ** vị trí bắt đầu.

Sau notebook này bạn sẽ:

1. Viết DP `f[t][i]` = tổng lớn nhất khi chọn **đúng** `t` đoạn trong `i` phần tử đầu.
2. Dùng tổng tiền tố `P` để viết chuyển trạng thái `f[t][i] = max(f[t][i−1], P[i] + max_{j ∈ [i−r, i−l]} (f[t−1][j] − P[j]))`.
3. Cài đặt **sliding window maximum** bằng `collections.deque`.
4. Viết 4 lời giải (vét cạn, DP duyệt độ dài, DP + hàng đợi đơn điệu, đệ quy có nhớ) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích `O(n · m)` thời gian và nhận ra vì sao phiên bản II (`n = 10^5`) cần ý tưởng mới.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` độ dài `n` và ba số nguyên `m`, `l`, `r`. Chọn từ `1` đến `m` mảng con (liên tiếp) **không chồng lấn**, mỗi mảng con có độ dài trong `[l, r]` (tính cả hai đầu). Tổng của tất cả các phần tử thuộc các mảng con được chọn phải **lớn nhất**. Trả về giá trị đó.

**Ví dụ 1:** `nums = [4,1,-5,2]`, `m = 2`, `l = 1`, `r = 3` → `7` (chọn `[4,1]` và `[2]`).

**Ví dụ 2:** `nums = [1,0,3,4]`, `m = 2`, `l = 1`, `r = 2` → `8` (chọn `[1]` và `[3,4]`).

**Ví dụ 3:** `nums = [-1,7,-4]`, `m = 1`, `l = 2`, `r = 3` → `6` (chọn `[-1,7]`).

**Ví dụ 4:** `nums = [-3,-4,-1]`, `m = 2`, `l = 1`, `r = 2` → `-1` (mọi mảng con đều âm; **phải** chọn ít nhất một nên chọn `[-1]`).

**Ràng buộc**

- `1 <= n == nums.length <= 1000`
- `-10^9 <= nums[i] <= 10^9`
- `1 <= m <= n`, `1 <= l <= r <= n`

Lưu ý: (1) **ít nhất một** đoạn, nên đáp án có thể âm (ví dụ 4); (2) vì `l ≥ 1` và `l ≤ n`, luôn chọn được ít nhất một đoạn; (3) "không giao nhau" cho phép hai đoạn **kề nhau** (đoạn này kết thúc ở `i−1`, đoạn kia bắt đầu ở `i`).

In [1]:
examples = [
    (dict(nums=[4, 1, -5, 2], m=2, l=1, r=3), 7),
    (dict(nums=[1, 0, 3, 4], m=2, l=1, r=2), 8),
    (dict(nums=[-1, 7, -4], m=1, l=2, r=3), 6),
    (dict(nums=[-3, -4, -1], m=2, l=1, r=2), -1),
]
for args, expected in examples:
    print("nums =", args["nums"], "| m, l, r =", (args["m"], args["l"], args["r"]), "| mong đợi:", expected)

nums = [4, 1, -5, 2] | m, l, r = (2, 1, 3) | mong đợi: 7
nums = [1, 0, 3, 4] | m, l, r = (2, 1, 2) | mong đợi: 8
nums = [-1, 7, -4] | m, l, r = (1, 2, 3) | mong đợi: 6
nums = [-3, -4, -1] | m, l, r = (2, 1, 2) | mong đợi: -1


## 3. Kiến thức nền

### 3.1. Tổng tiền tố `P`

`P[0] = 0`, `P[i] = nums[0] + ... + nums[i−1]`. Khi đó tổng mảng con chiếm các vị trí `j, j+1, ..., i−1` là `P[i] − P[j]`, và nó có độ dài `i − j`. Ta sẽ dùng "nửa mở" `[j, i)` để tránh lệch một đơn vị.

In [2]:
nums = [4, 1, -5, 2]
P = [0]
for x in nums:
    P.append(P[-1] + x)
print("P =", P)                                  # [0, 4, 5, 0, 2]
print("tổng nums[0:2] = P[2]-P[0] =", P[2] - P[0], "(độ dài", 2 - 0, ")")   # 5
print("tổng nums[3:4] = P[4]-P[3] =", P[4] - P[3], "(độ dài", 4 - 3, ")")   # 2

P = [0, 4, 5, 0, 2]
tổng nums[0:2] = P[2]-P[0] = 5 (độ dài 2 )
tổng nums[3:4] = P[4]-P[3] = 2 (độ dài 1 )


### 3.2. DP phân đoạn: trạng thái `f[t][i]`

`f[t][i]` = tổng lớn nhất khi chọn **đúng `t` đoạn** (mỗi đoạn dài `[l, r]`, không giao nhau) **chỉ dùng `i` phần tử đầu**. Hai khả năng cho phần tử `i − 1` (cuối cùng):

- không thuộc đoạn nào: `f[t][i − 1]`;
- là phần tử cuối của đoạn thứ `t`, đoạn này là `[j, i)` với `i − j ∈ [l, r]`: `f[t − 1][j] + P[i] − P[j]`.

Ta lấy `max` của hai. Khởi tạo `f[0][i] = 0` với mọi `i` (chọn 0 đoạn được 0 điểm). Các trạng thái **không thể** (ví dụ `t · l > i`) là `−∞`.

In [3]:
NEG = float("-inf")
print(max(NEG, -5), NEG + 10, NEG - 10, NEG == NEG)       # -5 -inf -inf True
print(max([NEG, NEG]))                                    # -inf: dãy toàn trạng thái không thể

-5 -inf -inf True
-inf


### 3.3. Cửa sổ trượt lớn nhất bằng hàng đợi đơn điệu

Bài toán "với mỗi `i`, lấy `max` của `a[i−r], ..., a[i−l]`" gọi là **sliding window maximum**. Dùng `collections.deque` lưu các **chỉ số** theo thứ tự giá trị **giảm dần**:

1. Khi thêm phần tử mới `a[j]`: bỏ các phần tử ở **cuối** deque có giá trị `≤ a[j]` (chúng không bao giờ là max nữa vì `a[j]` mới hơn và không nhỏ hơn).
2. Bỏ các chỉ số **đầu** deque nằm ngoài cửa sổ.
3. Phần tử **đầu** deque là max của cửa sổ.

Mỗi chỉ số vào ra deque tối đa một lần nên tổng `O(n)`.

In [4]:
from collections import deque

def sliding_max(a, lo_off, hi_off):
    """Với i = hi_off .. len(a)-1+lo_off... : max a[i-hi_off .. i-lo_off]; ví dụ demo cửa sổ [i-3, i-1]."""
    out = []
    dq = deque()                       # chứa chỉ số, giá trị giảm dần
    for i in range(len(a) + lo_off):
        j = i - lo_off                 # chỉ số mới nhất được phép vào cửa sổ
        if 0 <= j < len(a):
            while dq and a[dq[-1]] <= a[j]:
                dq.pop()
            dq.append(j)
        while dq and dq[0] < i - hi_off:
            dq.popleft()
        if dq:
            out.append((i, a[dq[0]]))
    return out

a = [3, 1, 4, 1, 5, 9, 2, 6]
print(sliding_max(a, 1, 3))           # i, max(a[i-3..i-1])
# kiểm tra bằng cách làm trực tiếp
for i, v in sliding_max(a, 1, 3):
    assert v == max(a[max(0, i - 3):i])
print("khớp vét cạn")

[(1, 3), (2, 3), (3, 4), (4, 4), (5, 5), (6, 9), (7, 9), (8, 9)]
khớp vét cạn


### 3.4. Biến đổi `P[i] + (f[t−1][j] − P[j])`

Với `i` cố định, hạng tử phụ thuộc `j` là `f[t−1][j] − P[j]`; `P[i]` là hằng số theo `j`. Vì vậy `max_j (f[t−1][j] + P[i] − P[j]) = P[i] + max_j (f[t−1][j] − P[j])`. Đó là sliding-window-max trên dãy `g[j] = f[t−1][j] − P[j]` với cửa sổ `j ∈ [i − r, i − l]`.

In [5]:
f_prev = [0, 0, 0, 0, 0]                      # f[0][*]
P = [0, 4, 5, 0, 2]
g = [f_prev[j] - P[j] for j in range(5)]
print("g =", g)
i, l, r = 4, 1, 3
window = list(range(max(0, i - r), i - l + 1))
print("cửa sổ j =", window, "-> max g =", max(g[j] for j in window), "-> f[1][4] ứng viên =", P[i] + max(g[j] for j in window))

g = [0, -4, -5, 0, -2]
cửa sổ j = [1, 2, 3] -> max g = 0 -> f[1][4] ứng viên = 2


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `nums = [4,1,-5,2]`, `m = 2`, `l = 1`, `r = 3`. `P = [0,4,5,0,2]`.

**Một đoạn (`t = 1`):** các mảng con dài `1..3`:

| Đoạn | Tổng |
|---|---|
| `[4]` | 4 |
| `[4,1]` | 5 |
| `[4,1,-5]` | 0 |
| `[1]` | 1 |
| `[1,-5]` | −4 |
| `[1,-5,2]` | −2 |
| `[-5]` | −5 |
| `[-5,2]` | −3 |
| `[2]` | 2 |

Tốt nhất một đoạn: `5` (`[4,1]`).

**Hai đoạn (`t = 2`):** `[4,1]` (5) + `[2]` (2) = `7`; hoặc `[4]` + `[2]` = 6; … Tốt nhất `7`. Vậy đáp án là `max(5, 7) = 7` ✓.

Đây là đúng cấu trúc DP: mỗi lần "thêm một đoạn" ta chọn điểm bắt đầu `j` cho đoạn cuối sao cho `f[t−1][j]` đủ lớn và `P[i] − P[j]` hợp lý — một bài toán tối ưu hai thành phần phụ thuộc cùng `j`.

## 5. Cách 1 — Vét cạn: thử mọi tập các đoạn

**Ý tưởng.** Đệ quy `go(i, left)`: ở vị trí `i` (các phần tử `< i` đã xử lý) còn được chọn thêm `left` đoạn nữa.

- bỏ qua phần tử `i`: `go(i + 1, left)`;
- bắt đầu một đoạn tại `i` với độ dài `len ∈ [l, r]` (`i + len ≤ n`): cộng `P[i + len] − P[i]` rồi `go(i + len, left − 1)`.

Trong code dưới đây ta viết đệ quy theo kiểu "chọn đoạn kế tiếp bắt đầu tại `start ≥ i`" và cập nhật đáp án mỗi khi đã chọn **ít nhất một** đoạn (`used ≥ 1`).

In [6]:
def brute_force(nums, m, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best = [None]

    def go(i, used, acc):
        if used >= 1 and (best[0] is None or acc > best[0]):
            best[0] = acc
        if used == m:
            return
        for start in range(i, n):                       # đoạn kế tiếp bắt đầu tại start >= i
            for ln in range(l, r + 1):
                if start + ln <= n:
                    go(start + ln, used + 1, acc + P[start + ln] - P[start])

    go(0, 0, 0)
    return best[0]

for args, expected in examples:
    got = brute_force(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** Số cách chọn tập đoạn không giao nhau bùng nổ (mũ theo `n`). Chỉ dùng cho `n ≤ 9` để đối chiếu.

**Nút thắt.** Nhiều tập đoạn khác nhau cùng "kết thúc" tại cùng vị trí `i` với cùng số đoạn `t`; phần còn lại phụ thuộc chỉ `(i, t)`. Cần ghi nhớ `f[t][i]`.

## 6. Bước đột phá

**Insight 1 — DP theo `(t, i)`.** `f[t][i]` như ở 3.2: tối ưu khi dùng `i` phần tử đầu và đúng `t` đoạn. Đáp án là `max_{1 ≤ t ≤ m} f[t][n]`.

**Insight 2 — tổng tiền tố.** Tổng đoạn cuối `[j, i)` là `P[i] − P[j]`, nên `f[t][i] = max(f[t][i−1], max_{j}(f[t−1][j] − P[j]) + P[i])` với `j ∈ [i − r, i − l]`, `j ≥ 0`.

**Insight 3 — cửa sổ trượt.** Với `t` cố định, khi `i` tăng thêm `1`, cửa sổ `[i − r, i − l]` dịch sang phải `1`: thêm `j = i − l`, bỏ `j = i − r − 1`. Dùng deque đơn điệu lấy `max` trong `O(1)` khấu hao ⇒ mỗi tầng `t` tốn `O(n)`, tổng `O(n · m)`.

**Insight 4 — chỉ cần hai tầng.** `f[t]` chỉ phụ thuộc `f[t−1]` và `f[t][i−1]`, nên chỉ giữ hai mảng (`prev`, `cur`) và cập nhật đáp án `best = max(best, cur[n])` sau mỗi tầng.

**Insight 5 — trạng thái không thể.** `f[t][i] = −∞` khi chưa thể xếp `t` đoạn vào `i` phần tử. Không đẩy các `j` có `f[t−1][j] = −∞` vào deque.

**Quy tắc "nếu… thì…":**

- Nếu `i < l` thì không có đoạn nào kết thúc tại `i` (không có `j` hợp lệ): `f[t][i] = f[t][i−1]`.
- Nếu `deque` rỗng (chưa có `j` hợp lệ, hoặc cửa sổ toàn `−∞`) thì chỉ có lựa chọn "bỏ qua phần tử `i − 1`".
- Chọn ít nhất 1 đoạn: lấy `max` qua `t ≥ 1`, **không** tính `t = 0` (nếu không đáp án âm sẽ bị thay bằng `0`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP `f[t][i]` duyệt mọi độ dài đoạn cuối (`O(n·m·(r−l+1))`)

**Giải thích từng đoạn.**

1. Dựng `P`. `f` là bảng `(m+1) × (n+1)`; `f[0][i] = 0`, `f[t][0] = −∞` với `t ≥ 1`.
2. Với `t = 1..m`, `i = 1..n`: `f[t][i] = f[t][i−1]`; rồi với mọi `ln ∈ [l, min(r, i)]` nếu `f[t−1][i−ln] > −∞`: `f[t][i] = max(f[t][i], f[t−1][i−ln] + P[i] − P[i−ln])`.
3. Đáp án `max_{t ≥ 1} f[t][n]`.

In [7]:
def dp_loop_lengths(nums, m, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    f = [[NEG] * (n + 1) for _ in range(m + 1)]
    for i in range(n + 1):
        f[0][i] = 0
    for t in range(1, m + 1):
        for i in range(1, n + 1):
            best = f[t][i - 1]
            for ln in range(l, min(r, i) + 1):
                prev = f[t - 1][i - ln]
                if prev > NEG:
                    cand = prev + P[i] - P[i - ln]
                    if cand > best:
                        best = cand
            f[t][i] = best
    return max(f[t][n] for t in range(1, m + 1))

for args, expected in examples:
    got = dp_loop_lengths(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** `O(n · m · (r − l + 1))` thời gian; `O(n · m)` bộ nhớ. Với `n = m = r = 1000` là `10^9` — quá chậm. Bước duyệt `ln` chính là cửa sổ `j ∈ [i − r, i − l]`: ta thay nó bằng hàng đợi đơn điệu.

### 7.2. Cách 3 — DP + hàng đợi đơn điệu, hai mảng cuộn (bản chính, `O(n·m)`)

**Giải thích từng đoạn.**

1. `prev = f[t−1]` (khởi tạo `prev = [0] * (n+1)` cho `t = 0`).
2. Với mỗi `t`: `cur[0] = −∞`; `dq` rỗng. Với `i = 1..n`:
   - `j = i − l`; nếu `j ≥ 0` và `prev[j] > −∞`: giá trị `v = prev[j] − P[j]`; bỏ phần tử cuối deque có `v' ≤ v`; thêm `(v, j)`.
   - bỏ các phần tử đầu deque có `j' < i − r`.
   - `cur[i] = cur[i−1]`; nếu deque không rỗng: `cur[i] = max(cur[i], P[i] + dq[0].v)`.
3. Sau tầng `t`: `best = max(best, cur[n])`; đặt `prev = cur`.

In [8]:
from collections import deque

def dp_deque(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    prev = [0] * (n + 1)                      # tầng t = 0
    best = NEG
    for t in range(1, m + 1):
        cur = [NEG] * (n + 1)
        dq = deque()                          # (v, j) với v giảm dần
        for i in range(1, n + 1):
            j = i - l
            if j >= 0 and prev[j] > NEG:
                v = prev[j] - P[j]
                while dq and dq[-1][0] <= v:
                    dq.pop()
                dq.append((v, j))
            while dq and dq[0][1] < i - r:
                dq.popleft()
            cur[i] = cur[i - 1]
            if dq and P[i] + dq[0][0] > cur[i]:
                cur[i] = P[i] + dq[0][0]
        if cur[n] > best:
            best = cur[n]
        prev = cur
    return best

for args, expected in examples:
    got = dp_deque(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** Mỗi tầng `t`: mỗi `j` vào và ra deque nhiều nhất một lần ⇒ `O(n)`; tổng `O(n · m)`; bộ nhớ `O(n)` (hai mảng). Với `n = m = 1000`: `10^6` bước deque — khoảng 1–2 giây trong Python.

### 7.3. Cách 4 — Đệ quy có nhớ (top-down) cho trực giác

Viết hàm `go(i, t)` = tổng lớn nhất từ các phần tử `i..n−1` khi cần chọn **đúng `t` đoạn nữa**:

- `t == 0`: trả `0`;
- bỏ qua phần tử `i`: `go(i + 1, t)`;
- bắt đầu đoạn tại `i` với độ dài `ln ∈ [l, r]`: `P[i + ln] − P[i] + go(i + ln, t − 1)`.

Đáp án `max_{t=1..m} go(0, t)`. Không dùng deque nên chậm hơn, nhưng dễ đọc và hữu ích để kiểm chứng.

In [9]:
from functools import lru_cache
import sys
sys.setrecursionlimit(10000)

def dp_topdown(nums, m, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)

    @lru_cache(maxsize=None)
    def go(i, t):
        if t == 0:
            return 0
        if i >= n:
            return NEG
        best = go(i + 1, t)
        for ln in range(l, r + 1):
            if i + ln <= n:
                sub = go(i + ln, t - 1)
                if sub > NEG:
                    best = max(best, P[i + ln] - P[i] + sub)
        return best

    return max(go(0, t) for t in range(1, m + 1))

for args, expected in examples:
    got = dp_topdown(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


**Độ phức tạp.** `O(n · m · (r − l + 1))` như cách 2 nhưng có chi phí đệ quy; chỉ dùng cho `n` nhỏ/vừa.

## 8. TRACE từng bước

Ta trace `dp_deque` cho ví dụ 1 (`nums = [4,1,-5,2]`, `m = 2`, `l = 1`, `r = 3`). In từng tầng `t`: mảng `prev`, và với mỗi `i`: `j` được thêm vào deque, deque sau khi dọn, `cur[i]`.

Bảng mong đợi `f[t][i]` (`i = 0..4`):

| `t` | `f[t][0..4]` |
|---|---|
| 0 | `[0, 0, 0, 0, 0]` |
| 1 | `[−∞, 4, 5, 5, 5]` |
| 2 | `[−∞, −∞, 5, 5, 7]` |

Giải thích ô `f[2][4] = 7`: đoạn thứ hai kết thúc tại `i = 4` (đoạn `[2]`, `j = 3`) cộng `f[1][3] = 5` (đoạn `[4,1]`): `5 + 2 = 7`. Đáp án `max(f[1][4], f[2][4]) = max(5, 7) = 7`.

In [10]:
def trace_deque(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    print("P =", P)
    prev = [0] * (n + 1)
    best = NEG
    for t in range(1, m + 1):
        print(f"\n===== tầng t = {t}, prev = {prev}")
        cur = [NEG] * (n + 1)
        dq = deque()
        for i in range(1, n + 1):
            j = i - l
            note = ""
            if j >= 0 and prev[j] > NEG:
                v = prev[j] - P[j]
                while dq and dq[-1][0] <= v:
                    dq.pop()
                dq.append((v, j))
                note = f"thêm j={j} (g={v})"
            while dq and dq[0][1] < i - r:
                dq.popleft()
            cur[i] = cur[i - 1]
            if dq and P[i] + dq[0][0] > cur[i]:
                cur[i] = P[i] + dq[0][0]
            print(f"i={i}: {note:18} deque={list(dq)} -> cur[{i}] = {cur[i]}")
        best = max(best, cur[n])
        print("f[%d] =" % t, cur)
        prev = cur
    print("\nđáp án =", best)
    return best

trace_deque([4, 1, -5, 2], 2, 1, 3)

P = [0, 4, 5, 0, 2]

===== tầng t = 1, prev = [0, 0, 0, 0, 0]
i=1: thêm j=0 (g=0)     deque=[(0, 0)] -> cur[1] = 4
i=2: thêm j=1 (g=-4)    deque=[(0, 0), (-4, 1)] -> cur[2] = 5
i=3: thêm j=2 (g=-5)    deque=[(0, 0), (-4, 1), (-5, 2)] -> cur[3] = 5
i=4: thêm j=3 (g=0)     deque=[(0, 3)] -> cur[4] = 5
f[1] = [-inf, 4, 5, 5, 5]

===== tầng t = 2, prev = [-inf, 4, 5, 5, 5]
i=1:                    deque=[] -> cur[1] = -inf
i=2: thêm j=1 (g=0)     deque=[(0, 1)] -> cur[2] = 5
i=3: thêm j=2 (g=0)     deque=[(0, 2)] -> cur[3] = 5
i=4: thêm j=3 (g=5)     deque=[(5, 3)] -> cur[4] = 7
f[2] = [-inf, -inf, 5, 5, 7]

đáp án = 7


7

In [11]:
# Trace ví dụ 4 (toàn số âm): bắt buộc chọn ít nhất 1 đoạn nên đáp án là -1, không phải 0
trace_deque([-3, -4, -1], 2, 1, 2)

P = [0, -3, -7, -8]

===== tầng t = 1, prev = [0, 0, 0, 0]
i=1: thêm j=0 (g=0)     deque=[(0, 0)] -> cur[1] = -3
i=2: thêm j=1 (g=3)     deque=[(3, 1)] -> cur[2] = -3
i=3: thêm j=2 (g=7)     deque=[(7, 2)] -> cur[3] = -1
f[1] = [-inf, -3, -3, -1]

===== tầng t = 2, prev = [-inf, -3, -3, -1]
i=1:                    deque=[] -> cur[1] = -inf
i=2: thêm j=1 (g=0)     deque=[(0, 1)] -> cur[2] = -7
i=3: thêm j=2 (g=4)     deque=[(4, 2)] -> cur[3] = -4
f[2] = [-inf, -inf, -7, -4]

đáp án = -1


-1

Ở ví dụ 4, tầng `t = 1` cho `f[1][3] = -1` (đoạn `[-1]`); tầng `t = 2` cho giá trị thấp hơn. Vì ta chỉ lấy `max` qua `t ≥ 1` nên đáp án là `-1`. Nếu ta (sai) tính cả `t = 0` thì đáp án sẽ là `0`.

## 9. Code nộp LeetCode

Dùng cách 3 (deque, hai mảng cuộn). Chữ ký đúng như skeleton (`List[int]`).

In [12]:
from collections import deque
from typing import List

class Solution:
    def maximumSum(self, nums: List[int], m: int, l: int, r: int) -> int:
        n = len(nums)
        NEG = float("-inf")
        P = [0] * (n + 1)                           # tổng tiền tố
        for i, x in enumerate(nums):
            P[i + 1] = P[i] + x
        prev = [0] * (n + 1)                        # f[0][*] = 0
        best = NEG
        for _ in range(m):                          # thêm lần lượt từng đoạn: t = 1..m
            cur = [NEG] * (n + 1)
            dq = deque()                            # (f[t-1][j] - P[j], j), giá trị giảm dần
            for i in range(1, n + 1):
                j = i - l                           # j mới được phép làm điểm bắt đầu của đoạn kết thúc ở i
                if j >= 0 and prev[j] > NEG:
                    v = prev[j] - P[j]
                    while dq and dq[-1][0] <= v:
                        dq.pop()
                    dq.append((v, j))
                while dq and dq[0][1] < i - r:      # bỏ j quá xa (độ dài > r)
                    dq.popleft()
                cur[i] = cur[i - 1]                 # phần tử i-1 không thuộc đoạn nào
                if dq and P[i] + dq[0][0] > cur[i]:
                    cur[i] = P[i] + dq[0][0]        # đoạn cuối là [j, i)
            if cur[n] > best:
                best = cur[n]
            prev = cur
        return best


sol = Solution()
for args, expected in examples:
    got = sol.maximumSum(**args)
    print(got, expected)
    assert got == expected

7 7
8 8
6 6
-1 -1


## 10. Vì sao thuật toán đúng

**Cấu trúc con tối ưu.** Xét cách chọn tối ưu cho `f[t][i]` (đúng `t` đoạn trong `i` phần tử đầu). Nếu phần tử `i−1` không thuộc đoạn nào thì phần còn lại là một cách chọn `t` đoạn trong `i−1` phần tử: tối ưu là `f[t][i−1]`. Nếu `i−1` thuộc đoạn cuối `[j, i)` thì các đoạn khác nằm hoàn toàn trong `j` phần tử đầu (không giao với đoạn cuối): phần còn lại là một cách chọn `t−1` đoạn trong `j` phần tử đầu, tối ưu là `f[t−1][j]`; tổng `= f[t−1][j] + P[i] − P[j]`. Vì mọi cách chọn rơi vào đúng một trong hai trường hợp, `f[t][i]` là `max` của chúng.

**Rút gọn cửa sổ.** Độ dài đoạn `i − j ∈ [l, r]` ⇔ `j ∈ [i − r, i − l]` (cùng `j ≥ 0`). Với `P[i]` cố định, tối đa hóa `f[t−1][j] + P[i] − P[j]` ⇔ tối đa hóa `g[j] = f[t−1][j] − P[j]` trên cửa sổ.

**Hàng đợi đơn điệu đúng.** Deque luôn giữ các chỉ số trong cửa sổ theo `g` giảm dần. Khi thêm `g` mới, các phần tử ở cuối có giá trị `≤ g` bị loại: chúng cũ hơn (sẽ rời cửa sổ sớm hơn) và không lớn hơn nên không bao giờ là max duy nhất của bất kỳ cửa sổ nào về sau. Phần tử đầu deque, sau khi bỏ các chỉ số quá hạn, là max của cửa sổ hiện tại.

**Trạng thái không thể.** Ta bỏ qua `j` có `prev[j] = −∞` nên không bao giờ dùng trạng thái không hợp lệ làm ứng viên.

**"Ít nhất một đoạn".** `best` chỉ lấy `cur[n]` từ `t ≥ 1`, tức không bao giờ là cách chọn rỗng. Đáp án có thể âm.

**`t` lớn hơn khả năng.** Nếu `t · l > n` thì `cur[n] = −∞` và không ảnh hưởng `best`; ít nhất `t = 1` có `cur[n] > −∞` vì `l ≤ n`.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| Vét cạn | mũ theo `n` | `O(n)` |
| DP duyệt độ dài | `O(n · m · (r − l + 1))` | `O(n · m)` |
| DP + deque (nộp) | `O(n · m)` | `O(n)` |
| Đệ quy có nhớ | `O(n · m · (r − l + 1))` | `O(n · m)` |

Với `n = m = 1000`: `10^6` bước deque. Phiên bản II (`n = 10^5`, `m ≤ n`) sẽ cần `10^{10}` ⇒ không dùng được cách này.

In [13]:
import time, random
rng = random.Random(2)
n = 1000
nums = [rng.randint(-10**9, 10**9) for _ in range(n)]
for (m, l, r) in [(1000, 1, 1000), (500, 1, 3), (100, 5, 50), (1000, 3, 1000)]:
    t0 = time.time()
    res = Solution().maximumSum(nums, m, l, r)
    print(f"m={m:>4}, l={l:>2}, r={r:>4}: {res}  ({time.time()-t0:.2f}s)")

m=1000, l= 1, r=1000: 252225229184  (0.21s)
m= 500, l= 1, r=   3: 252225229184  (0.13s)
m= 100, l= 5, r=  50: 181128896792  (0.03s)


m=1000, l= 3, r=1000: 209257616276  (0.13s)


## 12. Bộ test

Chiến lược test:

1. 4 ví dụ của đề (assert).
2. Biên: `n = 1`; `m = 1`; `l = r` (độ dài cố định); `l = r = n` (chỉ chọn cả mảng); toàn số âm; toàn số dương (nên dùng càng nhiều phần tử càng tốt); `t · l > n` cho một số tầng.
3. Phản ví dụ: không bắt buộc "ít nhất một đoạn" hoặc quên deque theo `l` (phần 13).
4. Đối chiếu vét cạn, DP duyệt độ dài, đệ quy có nhớ, DP + deque, `Solution` trên input ngẫu nhiên có seed cố định.

In [14]:
sol = Solution()
for args, expected in examples:
    assert sol.maximumSum(**args) == expected

assert sol.maximumSum([5], 1, 1, 1) == 5
assert sol.maximumSum([-5], 1, 1, 1) == -5                       # bắt buộc chọn
assert sol.maximumSum([1, 2, 3, 4], 4, 1, 1) == 10               # dùng cả 4 phần tử, mỗi đoạn một phần tử
assert sol.maximumSum([1, 2, 3, 4], 1, 4, 4) == 10               # l = r = n
assert sol.maximumSum([1, 2, 3, 4], 2, 2, 2) == 10               # hai đoạn dài 2
assert sol.maximumSum([1, -2, 3, -4, 5], 2, 1, 1) == 8           # hai đoạn một phần tử: 3 + 5
assert sol.maximumSum([-1, -2, -3], 3, 1, 1) == -1               # ít nhất một: chọn -1
print("test cố định OK")

test cố định OK


In [15]:
import random
rng = random.Random(777)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 8)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    l = rng.randint(1, n)
    r = rng.randint(l, n)
    m = rng.randint(1, n)
    expected = brute_force(nums, m, l, r)
    assert dp_loop_lengths(nums, m, l, r) == expected, (nums, m, l, r)
    assert dp_topdown(nums, m, l, r) == expected, (nums, m, l, r)
    assert dp_deque(nums, m, l, r) == expected, (nums, m, l, r)
    assert Solution().maximumSum(nums, m, l, r) == expected, (nums, m, l, r)
    cnt += 1
print("đã đối chiếu", cnt, "input ngẫu nhiên nhỏ: tất cả khớp vét cạn")

đã đối chiếu 1500 input ngẫu nhiên nhỏ: tất cả khớp vét cạn


In [16]:
# Input vừa (n tới 60): so DP duyệt độ dài với DP + deque
rng = random.Random(4321)
for _ in range(150):
    n = rng.randint(10, 60)
    nums = [rng.randint(-10**9, 10**9) for _ in range(n)]
    l = rng.randint(1, max(1, n // 4))
    r = rng.randint(l, n)
    m = rng.randint(1, n)
    e = dp_loop_lengths(nums, m, l, r)
    assert dp_deque(nums, m, l, r) == e == Solution().maximumSum(nums, m, l, r), (n, m, l, r)
print("DP duyệt độ dài == DP + deque == Solution trên 150 input vừa (giá trị tới 1e9)")

DP duyệt độ dài == DP + deque == Solution trên 150 input vừa (giá trị tới 1e9)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cho phép chọn 0 đoạn (lấy `t = 0` vào đáp án)

**Code sai:** `ans = max(f[t][n] for t in range(0, m + 1))`.
**Vì sao sai:** đề yêu cầu **ít nhất một** đoạn. Với mảng toàn âm đáp án phải âm, không phải `0`.
**Code đúng:** chỉ `t = 1..m`.

In [17]:
def wrong_allow_zero(nums, m, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    f = [[NEG] * (n + 1) for _ in range(m + 1)]
    for i in range(n + 1):
        f[0][i] = 0
    for t in range(1, m + 1):
        for i in range(1, n + 1):
            best = f[t][i - 1]
            for ln in range(l, min(r, i) + 1):
                if f[t - 1][i - ln] > NEG:
                    best = max(best, f[t - 1][i - ln] + P[i] - P[i - ln])
            f[t][i] = best
    return max(f[t][n] for t in range(0, m + 1))        # SAI: gồm t = 0

print("sai :", wrong_allow_zero(**examples[3][0]), " đúng:", Solution().maximumSum(**examples[3][0]))

sai : 0  đúng: -1


### Lỗi 2 — Lệch cửa sổ một đơn vị (`j ∈ [i−r, i−l]` viết thành `[i−r+1, i−l+1]` hoặc dùng `i` thay `i−1`)

**Code sai:** `j = i - l + 1` hoặc bỏ `j < i - r + 1`.
**Vì sao sai:** đoạn `[j, i)` có độ dài `i − j`; `j = i − l + 1` cho độ dài `l − 1 < l` (quá ngắn) và loại mất độ dài `r`.
**Code đúng:** `j = i - l` thêm vào, bỏ `j < i - r`.

In [18]:
def wrong_window_off_by_one(nums, m, l, r):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    prev = [0] * (n + 1)
    best = NEG
    for t in range(1, m + 1):
        cur = [NEG] * (n + 1)
        dq = deque()
        for i in range(1, n + 1):
            j = i - l + 1                                  # SAI
            if 0 <= j <= i and prev[j] > NEG:
                v = prev[j] - P[j]
                while dq and dq[-1][0] <= v:
                    dq.pop()
                dq.append((v, j))
            while dq and dq[0][1] < i - r + 1:             # SAI
                dq.popleft()
            cur[i] = cur[i - 1]
            if dq and P[i] + dq[0][0] > cur[i]:
                cur[i] = P[i] + dq[0][0]
        best = max(best, cur[n])
        prev = cur
    return best

rng = random.Random(10)
for _ in range(2000):
    n = rng.randint(2, 7)
    nums = [rng.randint(-5, 9) for _ in range(n)]
    l = rng.randint(1, n); r = rng.randint(l, n); m = rng.randint(1, n)
    if wrong_window_off_by_one(nums, m, l, r) != Solution().maximumSum(nums, m, l, r):
        print("phản ví dụ:", nums, (m, l, r), "sai:", wrong_window_off_by_one(nums, m, l, r), "đúng:", Solution().maximumSum(nums, m, l, r))
        break

phản ví dụ: [-5, 1, 2, 4, -5, -2] (3, 4, 5) sai: 7 đúng: 2


### Lỗi 3 — Dùng `0` thay `−∞` cho trạng thái không thể

**Code sai:** khởi tạo `cur = [0] * (n + 1)`.
**Vì sao sai:** `0` nghĩa là "chọn đúng `t` đoạn mà tổng bằng 0" — nhưng với `t ≥ 1` mà chưa đủ chỗ thì **không có cách chọn nào**. Tiếp tục dùng `0` có thể cho phép xếp `t` đoạn vào ít phần tử hơn khả năng hoặc làm đáp án quá cao.
**Code đúng:** `NEG = float("-inf")` và bỏ qua `j` có `prev[j] = −∞`.

### Lỗi 4 — Quên rằng hai đoạn có thể kề nhau

**Code sai:** bắt đoạn kế tiếp bắt đầu ở `j + 1` (chừa một ô trống).
**Vì sao sai:** đề chỉ cấm **chồng lấn**; đoạn `[0,2)` và `[2,4)` là hợp lệ. Trong DP, `f[t−1][j]` dùng các phần tử `< j` và đoạn mới bắt đầu ngay tại `j` (nửa mở), nên đã cho phép kề nhau.

## 14. Mẹo & nhận dạng pattern

- **"Chia mảng thành tối đa `m` đoạn với ràng buộc độ dài, tối đa hóa tổng"** → DP `f[t][i]` + tổng tiền tố.
- Chuyển động `max_j (f[t−1][j] + P[i] − P[j])` = `P[i] + max_j (f[t−1][j] − P[j])`: tách hạng tử phụ thuộc `j` ⇒ **sliding window maximum** (deque) hoặc cây phân đoạn.
- Cửa sổ `j ∈ [i − r, i − l]` dịch đều theo `i` ⇒ deque đơn điệu là công cụ chuẩn.
- Dùng `−∞` cho trạng thái không hợp lệ; chọn `t ≥ 1` ở đáp án khi đề bắt buộc "ít nhất một".
- Khi `n` lớn (`10^5`) và `m` cũng lớn, `O(nm)` không đủ: nghĩ đến **tính lồi** của `f(t)` theo `t` và kỹ thuật **Aliens trick** (đó chính là bài II).
- Vét cạn ở `n ≤ 8` là cách rất tốt để kiểm tra chỉ số lệch một đơn vị.

## 15. Biến thể

1. **Phiên bản II** (`n = 10^5`, `m ≤ n`): cần Aliens trick; xem notebook LeetCode 3957.
2. **Đúng `m` đoạn** (không phải "tối đa"): lấy `f[m][n]` (Bài 5).
3. **Không giới hạn độ dài** (`l = 1`, `r = n`): bài *Maximum Sum of `k` Non-overlapping Subarrays*; DP `O(nk)` đơn giản hơn không cần deque.
4. **Các đoạn phải cách nhau ít nhất `g` ô:** thay `f[t−1][j]` bằng `f[t−1][j − g]`.
5. **Mỗi đoạn tốn một khoản phí cố định `c`:** thay tổng bằng `tổng − c` mỗi đoạn; đây gần như là bước trong Aliens trick.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tập đoạn | mũ | `O(n)` | đối chiếu `n ≤ 9` |
| 2. DP duyệt độ dài | `f[t][i]` với vòng `ln ∈ [l, r]` | `O(n·m·(r−l+1))` | `O(n·m)` | hiểu bài, `n` nhỏ |
| 3. DP + deque | cửa sổ trượt max trên `f[t−1][j] − P[j]` | `O(n·m)` | `O(n)` | **nộp bài** |
| 4. Top-down | `go(i, t)` có nhớ | `O(n·m·(r−l+1))` | `O(n·m)` | trực giác / kiểm chứng |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Với `nums = [4, 1, -5, 2]`, tổng tiền tố `P` là gì? Tổng mảng con `nums[1:3]` (hai phần tử `1, -5`) tính bằng `P` ra bao nhiêu?

**Bài 2 (tính tay).** `nums = [2, -1, 3]`, `m = 2`, `l = 1`, `r = 2`. Đáp án là bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thiện `window_max(g, i, lo_off, hi_off)` trả về `max` của `g[j]` với `j ∈ [i − hi_off, i − lo_off]`, `j ≥ 0` (cài bằng vòng lặp thường, không cần deque), hoặc `None` nếu rỗng.

**Bài 4 (tự viết lại).** Viết `my_dp(nums, m, l, r)` theo cách 2 (duyệt độ dài) mà không nhìn code.

**Bài 5 (biến thể).** Viết `exact_m(nums, m, l, r)`: tổng lớn nhất khi chọn **đúng** `m` đoạn; trả `None` nếu không thể (khi `m · l > n`).

In [19]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def window_max(g, i, lo_off, hi_off):
    # TODO: duyệt j từ max(0, i - hi_off) tới i - lo_off, trả max g[j]; None nếu cửa sổ rỗng
    pass

# Bài 4 — TODO
def my_dp(nums, m, l, r):
    # TODO: P, f[t][i], đáp án max_{t>=1} f[t][n]
    pass

# Bài 5 — TODO
def exact_m(nums, m, l, r):
    # TODO: như my_dp nhưng trả về f[m][n] (None nếu -inf)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `P[0] = 0`, mỗi bước cộng thêm một phần tử.
- Mức 2: `P = [0, 4, 5, 0, 2]`.
- Mức 3: `nums[1:3]` ứng với `P[3] − P[1] = 0 − 4 = −4` (đúng: `1 + (−5) = −4`).

**Bài 2.**
- Mức 1: thử chọn hai đoạn một phần tử.
- Mức 2: `[2]` và `[3]` cho `5`; hoặc `[2,-1]` (1) và `[3]` (3) cho `4`.
- Mức 3: `5`.

**Bài 3.**
- Mức 1: cửa sổ là `i−hi_off ≤ j ≤ i−lo_off`.
- Mức 2: nhớ giới hạn dưới `0`.
- Mức 3: nếu `lo > hi` trả `None`.

**Bài 4.**
- Mức 1: `f[0][i] = 0`, `f[t][0] = -inf`.
- Mức 2: vòng `ln in range(l, min(r, i) + 1)`.
- Mức 3: `max(f[t][i-1], f[t-1][i-ln] + P[i] - P[i-ln])`.

**Bài 5.**
- Mức 1: chỉ khác bước lấy đáp án.
- Mức 2: `f[m][n]`.
- Mức 3: nếu `-inf` trả `None`.

## 19. Đáp án tham khảo

In [20]:
# Bài 1
P = [0]
for x in [4, 1, -5, 2]:
    P.append(P[-1] + x)
assert P == [0, 4, 5, 0, 2] and P[3] - P[1] == -4
# Bài 2
assert Solution().maximumSum([2, -1, 3], 2, 1, 2) == 5 == brute_force([2, -1, 3], 2, 1, 2)

# Bài 3
def window_max_ans(g, i, lo_off, hi_off):
    lo = max(0, i - hi_off)
    hi = i - lo_off
    if lo > hi or hi < 0:
        return None
    return max(g[lo:hi + 1])

g = [5, -1, 7, 3, 2]
assert window_max_ans(g, 4, 1, 3) == 7        # j in [1, 3]
assert window_max_ans(g, 1, 2, 3) is None     # j in [-2, -1] -> rỗng
assert window_max_ans(g, 2, 1, 5) == 5        # j in [0, 1]
print("Bài 1–3 OK")

Bài 1–3 OK


In [21]:
# Bài 4
def my_dp_ans(nums, m, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    f = [[NEG] * (n + 1) for _ in range(m + 1)]
    for i in range(n + 1):
        f[0][i] = 0
    for t in range(1, m + 1):
        for i in range(1, n + 1):
            best = f[t][i - 1]
            for ln in range(l, min(r, i) + 1):
                if f[t - 1][i - ln] > NEG:
                    best = max(best, f[t - 1][i - ln] + P[i] - P[i - ln])
            f[t][i] = best
    return f

def my_dp_best(nums, m, l, r):
    f = my_dp_ans(nums, m, l, r)
    return max(f[t][len(nums)] for t in range(1, m + 1))

# Bài 5: đúng m đoạn
def exact_m_ans(nums, m, l, r):
    v = my_dp_ans(nums, m, l, r)[m][len(nums)]
    return None if v == NEG else v

def brute_exact(nums, m, l, r):
    n = len(nums)
    P = [0]
    for x in nums:
        P.append(P[-1] + x)
    best = [None]
    def go(i, used, acc):
        if used == m and (best[0] is None or acc > best[0]):
            best[0] = acc
        if used == m:
            return
        for s in range(i, n):
            for ln in range(l, r + 1):
                if s + ln <= n:
                    go(s + ln, used + 1, acc + P[s + ln] - P[s])
    go(0, 0, 0)
    return best[0]

rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 8)
    nums = [rng.randint(-9, 9) for _ in range(n)]
    l = rng.randint(1, n); r = rng.randint(l, n); m = rng.randint(1, n)
    assert my_dp_best(nums, m, l, r) == Solution().maximumSum(nums, m, l, r)
    assert exact_m_ans(nums, m, l, r) == brute_exact(nums, m, l, r), (nums, m, l, r)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP phân đoạn + cửa sổ trượt (deque):**

```python
P = tổng tiền tố
prev = [0] * (n + 1)                 # t = 0
best = -inf
for t in range(1, m + 1):
    cur = [-inf] * (n + 1); dq = deque()
    for i in range(1, n + 1):
        j = i - l                    # j mới vào cửa sổ
        if j >= 0 and prev[j] > -inf:
            v = prev[j] - P[j]
            while dq and dq[-1][0] <= v: dq.pop()
            dq.append((v, j))
        while dq and dq[0][1] < i - r: dq.popleft()
        cur[i] = cur[i - 1]
        if dq: cur[i] = max(cur[i], P[i] + dq[0][0])
    best = max(best, cur[n]); prev = cur
return best
```

**Checklist trước khi submit:**

- [ ] `f[t][i]` = đúng `t` đoạn trong `i` phần tử đầu; `f[0][*] = 0`, trạng thái không thể là `−∞`.
- [ ] Cửa sổ `j ∈ [i − r, i − l]` (không lệch một đơn vị), đoạn `[j, i)` nửa mở.
- [ ] Deque lưu `f[t−1][j] − P[j]` giảm dần; bỏ `j` quá hạn ở đầu.
- [ ] Đáp án = `max` qua `t = 1..m` (ít nhất một đoạn), không gồm `t = 0`.
- [ ] Đáp án có thể âm.
- [ ] Hai đoạn được phép kề nhau.
- [ ] Đã đối chiếu vét cạn trên mảng `n ≤ 8`.